## SOTA RAG with EquoAI!

In [1]:
! pip install equoai 
! pip install PyPDF2
! pip install sentence-transformers
! pip install ollama 
! pip install requests 

In [157]:
from equoai import equonode
from sentence_transformers import SentenceTransformer
import os 
import PyPDF2
import numpy as np
from ollama import Client
import sys
import requests

def query_with_ollama(prompt : str, is_verbose=True) -> str:
    '''Query function using Ollama proxy'''

    # client = Client(host=os.getenv("LOCAL_SERVER_URI"))
    client = Client(host="https://ollama.newatlantis.top")

    # client = Client(host="https://pop-os.tailcff25c.ts.net/abcdefghijk")

    stream = client.chat( 
        model = "llama3.2:latest",

        # model = "mistral:7b-instruct", #This has to be added to the hardware we're using.
        messages=[{'role': 'user', 'content': prompt}],
        stream=True,
    )
    completion=""
    try:
        for chunk in stream:
            if chunk is not None:
                if is_verbose is True:
                    print(chunk['message']['content'])
                completion += chunk['message']['content']
                sys.stdout.flush()
                if chunk['message']['content'] == "<|eot_id|>":
                    stream.close() #Close the stream?
                    print("Closing the stream.")

    except Exception as e:
        print(f"{e}") 

    # return request.data["message"
    return completion 




class RAGPipeline:
    
    def __init__(self, model_name="sentence-transformers/multi-qa-MiniLM-L6-cos-v1"):
        self.model = SentenceTransformer(model_name)
        self.documents = []
        self.context = []
        self.entity_index = []
        
    def cosine(self, u: np.ndarray, v: np.ndarray) -> float:
        """
        Cosine similarity metric
        """
        return u.dot(v) / np.sqrt(u.dot(u) * v.dot(v))
    
    def process(self, query: str, documents: list[str]) -> None:
        """
        Order pieces of context by relevance to the user's query
        """
        # Generate embeddings 
        x = self.model.encode(query)
        vectors = self.model.encode(documents)
        self.context = [{"text":doc, "score":self.cosine(vectors[i], x)} for i, doc in enumerate(documents)]
        self.context= sorted(self.context, key=lambda x: x["score"], reverse=True)
        
        
    def retrieve(self, k=10) -> str:
        """
        Retrieve top-K most relevant documents
        Format: 
        Article 1: blah blah blah 
        Article 2: ...
        Article 3: ...
        """
        return "document: ".join([f'{i}: {obj["text"]} 'for i, obj in enumerate(self.context)][:k])
    
    def run(self, query: str, documents: list[str], is_optimized=False) -> str:
        """
            Handle the entire RAG, end-to-end
        """
        pipeline.process(query, documents)
        context = pipeline.retrieve()

        return query_with_ollama(f"Article: {context} \n Answer the following question using the article provided: {query}",
                 is_verbose=False)

    
def parse_pdf(file_path):
    """
    Read the local PDF file
    """
    with open(file_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = ""
        
        # Iterate over all the pages and extract text
        for page_num in range(len(reader.pages)):
            page = reader.pages[page_num]
            text += page.extract_text()
            
        return text




In [158]:
pdf_file_path = os.path.join(os.getcwd(), 'smol.pdf')
pdf_text = parse_pdf(pdf_file_path)
document_contents = pdf_text.split(".")
project_name="esg-rag"
equo = equonode(project_name)
db = equonode(None)

In [162]:
# Model for generating Q-A embeddings


#Obtain embeddings and convert from ndarray to Python list. 
#Make sure that your embeddings are a list of floating point values before uploading
embeddings = pipeline.model.encode(document_contents)
embeddings = embeddings.tolist()
project_name='esg-report:power-corporation'

In [163]:
query = 'How has the leadership of the corporation changed since Paul Desmarais stepped down?'

pipeline = RAGPipeline()

# pipeline.process(query, document_contents)
# context = pipeline.retrieve()

# answer = query_with_ollama(f"Article: {context} \n Answer the following question using the article provided: {query}",
#                  is_verbose=False)

answer = pipeline.run(query, document_contents)



In [164]:
# query = 'What people have historically inhabited the nation of Kazakhstan?'
query = 'How has the leadership of the corporation changed since Paul Desmarais stepped down?'
query = model.encode(query)#.tolist()
 
context_text = "".join([obj["text"] for obj in pipeline.context][:10])
snippets = context_text.split(".")
context = pipeline.model.encode(snippets).tolist()

(best_document, highest_similarity, dp) = db.dynamic_retrieval(
                                            query, 
                                            context, 
                                            snippets, 
                                            optimized=False,
                                            threshold=0.0
                                        )

In [165]:
best_document, highest_similarity

(" and André Desmarais from their executive roles as Co-Chief \nExecutive Officers of the Corporation on February\xa013, 2020, the Governance and Nominating Committee \nis now entirely composed of Directors who are not members of management of the Corporation F ollowing the retirement of Paul Desmarais,\xa0Jr, Chairman, and André Desmarais, Deputy Chairman, being former executive officers \nof the Corporation within the past three years, are not independent T he Corporation believes that continuity of membership is critical to its Board’s efficient operation and \naccordingly has not adopted policies imposing an arbitrary term or retirement age limit for its Directors In such a context,\xa0the \nCorporation believes that a lengthy Board tenure, not limited by arbitrary determinations, is vital to \nthe\xa0Directors’ understanding of the Corporation’s diverse businesses and those of its group companies, \nand to their bringing a substantive contribution to the Board The \nSustainability

### Anonymization API Demo 

#### Here, we are going to keep track of a handful of documents, anonymize their contents, 
#### and keep track of names entities along the way.
#### This allows us to do nearly anything with Generative AI, while remaining compliant in regard to data privacy and standards such as GDPR and SOC-2. 


In [150]:
from dotenv import load_dotenv
# load_dotenv()
# URI = os.getenv("URI")

URI="https://api-equo-ai.tail44cf99.ts.net"

def anonymize(text: str, entities={}, person_count=0, org_count=0, location_count=0) -> str:
    """
    Anonymize your documents using our API for data protection!
    API keys will become available for anyone signing up here:

    https://equo.ai/signup
    """
    r = requests.post(f"{URI}/protect/",
                      json={
                            "text":text,
                            "access_token":"abdefg", 
                            "entities":entities,
                            "item_counts":{
                                "person_count":person_count,
                                "org_count":org_count,
                                "location_count":location_count
                        }
    })
    return r.json()



def track_entities(doc, entities):
    """
    Keep track of previously 
    seen named entities 
    """
    for ent in list(entities.keys()):
    #     print(result["entities"][ent])
        try:
            category = entities[ent]
            doc = doc.replace(ent, category)
        except Exception as e:
            print(e)
    # print(list(result["entities"].keys()))
    return doc 

result = anonymize("Roger and Donald both work for Acme Inc. Their father worked at IBM.")

person_count = result["num_persons"] 
org_count = result["num_orgs"], 
location_count = result["num_locations"]

q2 = "David and Roger are brothers."
print(result["entities"])
entities = result["entities"]

q2 = track_entities(q2, entities)
next_result = anonymize(q2, entities, person_count, org_count, location_count)
print(next_result["text"])

{'Roger': '<PERSON 1>', 'Donald': '<PERSON 2>', 'Acme Inc.': '<ORGANIZATION 1>', 'IBM': '<ORGANIZATION 2>'}
<PERSON 3> and <PERSON 1> are brothers.


In [160]:
entities = {}
person_count = 0
org_count = 0
location_count = 0
anonymized_documents = []

documents = [
    "David and Roger are brothers. Devin is their cousin",
    "Roger and Donald both work for Acme Inc. Their father worked at IBM.",
    "Donald and David are actually best friends. Sometimes they hang out with Roger.",
    "David and Roger are brothers",
    "Donald and Roger sometimes hang out.",
    "David does not talk to his cousin"

]

for i, doc in enumerate(documents):
    try:
        doc = track_entities(doc, entities)
        result = anonymize(doc, entities, person_count, org_count, location_count)
        person_count = result["num_persons"] 
        org_count = result["num_orgs"], 
        location_count = result["num_locations"]
        entities = result["entities"]
        if i > 1:
            anonymized_documents.append(doc)
        print(f'Document: {doc}')
    except Exception as e:
        print(e)

Document: David and Roger are brothers. Devin is their cousin
Expecting value: line 2 column 1 (char 1)
Document: Donald and <PERSON 1> are actually best friends. Sometimes they hang out with <PERSON 2>.
Document: <PERSON 1> and <PERSON 2> are brothers
Document: <PERSON 4> and <PERSON 2> sometimes hang out.
Document: <PERSON 1> does not talk to his cousin


In [152]:
context = "".join(anonymized_documents[2:])
# context = anonymized_documents[-1]

question = "Who sometimes hang out together?"
query = f"Article: {context} \n Given the article above, answer the following question: {question}"
ans = query_with_ollama(query, is_verbose=False)

#The LLM is smart enough to keep track of the protected named entities if we are!
print(ans)

According to the article, PERSON 4 and PERSON 2 sometimes hang out together.


## We've learned how to do two things:
#### Setup a RAG Pipeline 